Splitting the tasks into three categories and saving them into different csv files


In [9]:
import pandas as pd
import re #regular expression
from collections import defaultdict
label_column="class"
expected_features_per_task=18
copy_idx     = {6,7,8,9,10,11,12,13,15,16,17,19,22,25}
graphic_idx  = {2,3,4,5,21,24}
memory_idx   = {1,14,18,20,23}
df=pd.read_csv("data.csv")
all_cols=list(df.columns)
if label_column not in all_cols:
  raise ValueError(f"{label_column} not in columns")
def cols_for_task(task_num,columns):#returns list of columns that end with task number using regex expression
  pattern=re.compile(rf'(?<!\d){re.escape(str(task_num))}$')
  matches=[c for c in columns if pattern.search(c)]
  return matches
# build dataset for a set of task numbers
def build_task_dataset(task_set,df,expected_per_task=expected_features_per_task):
  task_set=sorted(task_set)
  selected_cols=[]
  per_task_report=defaultdict(list)
  for t in task_set:
    cols=cols_for_task(t,df.columns)
    cols_sorted=sorted(cols)
    per_task_report[t]=cols_sorted
    selected_cols.extend(cols_sorted)
#remove duplicates while preserving the order
  seen=set()
  selected_cols_unique=[]
  for c in selected_cols:
    if c not in seen:
      seen.add(c)
      selected_cols_unique.append(c)
  #prepare final dataframe and put label column at the end
  if not selected_cols_unique:
    raise ValueError("No columns found")
  final_cols=selected_cols_unique+[label_column]
  missing=[c for c in final_cols if c not in df.columns]
  if missing:
    raise RuntimeError(f"Missing columns: {missing}")
  return df[final_cols].copy(),per_task_report
#build and save the datasets
df_copy_tasks,report_copy=build_task_dataset(copy_idx,df)
df_graphic_tasks,report_graphic=build_task_dataset(graphic_idx,df)
df_memory_tasks,report_memory=build_task_dataset(memory_idx,df)
#save to csv
df_copy_tasks.to_csv("copy_tasks.csv",index=False)
df_graphic_tasks.to_csv("graphic_tasks.csv",index=False)
df_memory_tasks.to_csv("memory_tasks.csv",index=False)
#X,y ready for SVC
X_copy=df_copy_tasks.drop(columns=[label_column]).values
y_copy=df_copy_tasks[label_column].values
X_graphic=df_graphic_tasks.drop(columns=[label_column]).values
y_graphic=df_graphic_tasks[label_column].values
X_memory=df_memory_tasks.drop(columns=[label_column]).values
y_memory=df_memory_tasks[label_column].values




SVC on Copy tasks

In [11]:
from sklearn.model_selection import GridSearchCV,ShuffleSplit,train_test_split #for hyperparameter search
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
import numpy as np
outer_cv=ShuffleSplit(n_splits=20,test_size=0.2,random_state=19)
pipe_svc_copy=Pipeline([('scaler1',StandardScaler()),('pca',PCA(n_components=24,random_state=42)),('scaler2',StandardScaler()),('svc',SVC(random_state=42))])
param_grid_svc_copy={'svc__kernel':['linear','poly','rbf','sigmoid'],
                'svc__C':[0.1,1,10,100],
                'svc__gamma':['scale',1,0.1,0.01,0.001],
                'svc__tol':[1e-3,1e-4,1e-5]}
acc_svc_tuned_copy=[]
for i,(trainval_idx,test_idx) in enumerate(outer_cv.split(X_copy,y_copy)):
    X_trainval,X_test=X_copy[trainval_idx],X_copy[test_idx]
    y_trainval,y_test=y_copy[trainval_idx],y_copy[test_idx]
    X_train,X_val,y_train,y_val=train_test_split(X_trainval,y_trainval,test_size=0.25,stratify=y_trainval,random_state=42)# further splitting
    grid_svc_copy=GridSearchCV( estimator=pipe_svc_copy, #model to tune
                          param_grid=param_grid_svc_copy, # parameters to try
                          cv=3, #no of folds
                          scoring='accuracy', #how to measure the performace
                          n_jobs=-1 #all cpu cores to speed the training
                         ) #grid search on only train vs val
    grid_svc_copy.fit(X_train,y_train)
    best_model=grid_svc_copy.best_estimator_
    best_model.fit(np.vstack([X_train,X_val]),np.concatenate([y_train,y_val])) #joins two part to form train+val set
    acc=best_model.score(X_test,y_test)
    acc_svc_tuned_copy.append(acc)
    print(f"split{i+1}:{acc:.2f}")
mean_acc_svc_copy=np.mean(acc_svc_tuned_copy)*100
std_acc_svc_copy=np.std(acc_svc_tuned_copy)*100
print(f"\n mean accuracy (copy tasks):{mean_acc_svc_copy:.2f}")
print(f"\n std accuracy (copy tasks):{std_acc_svc_copy:.2f}")





split1:0.71
split2:0.83
split3:0.77
split4:0.94
split5:0.89
split6:0.83
split7:0.89
split8:0.83
split9:0.74
split10:0.83
split11:0.80
split12:0.97
split13:0.74
split14:0.83
split15:0.80
split16:0.83
split17:0.86
split18:0.91
split19:0.89
split20:0.74

 mean accuracy (copy tasks):83.14

 std accuracy (copy tasks):6.76


SVC on graphics tasks

In [12]:
from sklearn.model_selection import GridSearchCV,ShuffleSplit,train_test_split #for hyperparameter search
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
import numpy as np
outer_cv=ShuffleSplit(n_splits=20,test_size=0.2,random_state=19)
pipe_svc_graphic=Pipeline([('scaler1',StandardScaler()),('pca',PCA(n_components=24,random_state=42)),('scaler2',StandardScaler()),('svc',SVC(random_state=42))])
param_grid_svc_graphic={'svc__kernel':['linear','poly','rbf','sigmoid'],
                'svc__C':[0.1,1,10,100],
                'svc__gamma':['scale',1,0.1,0.01,0.001],
                'svc__tol':[1e-3,1e-4,1e-5]}
acc_svc_tuned_graphic=[]
for i,(trainval_idx,test_idx) in enumerate(outer_cv.split(X_graphic,y_graphic)):
    X_trainval,X_test=X_graphic[trainval_idx],X_graphic[test_idx]
    y_trainval,y_test=y_graphic[trainval_idx],y_graphic[test_idx]
    X_train,X_val,y_train,y_val=train_test_split(X_trainval,y_trainval,test_size=0.25,stratify=y_trainval,random_state=42)# further splitting
    grid_svc_graphic=GridSearchCV( estimator=pipe_svc_graphic, #model to tune
                          param_grid=param_grid_svc_graphic, # parameters to try
                          cv=3, #no of folds
                          scoring='accuracy', #how to measure the performace
                          n_jobs=-1 #all cpu cores to speed the training
                         ) #grid search on only train vs val
    grid_svc_graphic.fit(X_train,y_train)
    best_model=grid_svc_graphic.best_estimator_
    best_model.fit(np.vstack([X_train,X_val]),np.concatenate([y_train,y_val])) #joins two part to form train+val set
    acc=best_model.score(X_test,y_test)
    acc_svc_tuned_graphic.append(acc)
    print(f"split{i+1}:{acc:.2f}")
mean_acc_svc_graphic=np.mean(acc_svc_tuned_graphic)*100
std_acc_svc_graphic=np.std(acc_svc_tuned_graphic)*100
print(f"\n mean accuracy (graphic tasks):{mean_acc_svc_graphic:.2f}")
print(f"\n std accuracy (graphic tasks):{std_acc_svc_graphic:.2f}")





split1:0.74
split2:0.77
split3:0.74
split4:0.74
split5:0.74
split6:0.71
split7:0.86
split8:0.74
split9:0.77
split10:0.74
split11:0.83
split12:0.77
split13:0.80
split14:0.86
split15:0.77
split16:0.77
split17:0.80
split18:0.74
split19:0.83
split20:0.77

 mean accuracy (graphic tasks):77.57

 std accuracy (graphic tasks):3.97


SVC on memory tasks

In [17]:
from sklearn.model_selection import GridSearchCV,ShuffleSplit,train_test_split #for hyperparameter search
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
import numpy as np
outer_cv=ShuffleSplit(n_splits=20,test_size=0.2,random_state=30)
pipe_svc_memory=Pipeline([('scaler1',StandardScaler()),('pca',PCA(n_components=24,random_state=42)),('scaler2',StandardScaler()),('svc',SVC(random_state=42))])
param_grid_svc_memory={'svc__kernel':['linear','poly','rbf','sigmoid'],
                'svc__C':[0.1,1,10,100],
                'svc__gamma':['scale',1,0.1,0.01,0.001],
                'svc__tol':[1e-3,1e-4,1e-5]}
acc_svc_tuned_memory=[]
for i,(trainval_idx,test_idx) in enumerate(outer_cv.split(X_memory,y_memory)):
    X_trainval,X_test=X_memory[trainval_idx],X_memory[test_idx]
    y_trainval,y_test=y_memory[trainval_idx],y_memory[test_idx]
    X_train,X_val,y_train,y_val=train_test_split(X_trainval,y_trainval,test_size=0.25,stratify=y_trainval,random_state=42)# further splitting
    grid_svc_memory=GridSearchCV( estimator=pipe_svc_memory, #model to tune
                          param_grid=param_grid_svc_memory, # parameters to try
                          cv=3, #no of folds
                          scoring='accuracy', #how to measure the performace
                          n_jobs=-1 #all cpu cores to speed the training
                         ) #grid search on only train vs val
    grid_svc_memory.fit(X_train,y_train)
    best_model=grid_svc_memory.best_estimator_
    best_model.fit(np.vstack([X_train,X_val]),np.concatenate([y_train,y_val])) #joins two part to form train+val set
    acc=best_model.score(X_test,y_test)
    acc_svc_tuned_memory.append(acc)
    print(f"split{i+1}:{acc:.2f}")
mean_acc_svc_memory=np.mean(acc_svc_tuned_memory)*100
std_acc_svc_memory=np.std(acc_svc_tuned_memory)*100
print(f"\n mean accuracy (memory tasks):{mean_acc_svc_memory:.2f}")
print(f"\n std accuracy (memory tasks):{std_acc_svc_memory:.2f}")





split1:0.69
split2:0.71
split3:0.83
split4:0.60
split5:0.74
split6:0.66
split7:0.60
split8:0.83
split9:0.74
split10:0.69
split11:0.80
split12:0.71
split13:0.66
split14:0.69
split15:0.63
split16:0.71
split17:0.83
split18:0.74
split19:0.71
split20:0.80

 mean accuracy (memory tasks):71.86

 std accuracy (memory tasks):7.01


Final Evaluation

In [26]:
print("Classical\n")
print("Copy","Graphics","Memory",sep="\t\t")
print(
    f"{mean_acc_svc_copy:.2f} ± {std_acc_svc_copy:.2f}",
    f"{mean_acc_svc_graphic:.2f} ± {std_acc_svc_graphic:.2f}",
    f"{mean_acc_svc_memory:.2f} ± {std_acc_svc_memory:.2f}",
    sep="\t"
)


Classical

Copy		Graphics		Memory
83.14 ± 6.76	77.57 ± 3.97	71.86 ± 7.01
